# Vietnamese ASR (Zipformer) - Module Build and Realtime Demo

Notebook này triển khai và demo module ASR tiếng Việt theo yêu cầu:
- Có `test_inference()` cho file audio.
- Có `realtime_inference()` cho microphone.
- Có phần cập nhật `requirements.txt` để thêm dependency còn thiếu.

Lưu ý:
- Notebook này tạo thêm skeleton trong `src/asr_module/` để minh họa kiến trúc.
- Code production đã được đặt trong `modules/asr_zipformer.py` để dùng ngay trong project.

## 1) Install and Verify Dependencies

In [ ]:
%pip install -q sherpa-onnx sounddevice librosa numpy scipy torch transformers

In [ ]:
import importlib
import os
import platform
from pathlib import Path

import numpy as np
import scipy
import torch

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

for pkg in ['sherpa_onnx', 'sounddevice', 'librosa']:
    try:
        m = importlib.import_module(pkg)
        print(f'{pkg}: OK ({getattr(m, "__version__", "no_version")})')
    except Exception as e:
        print(f'{pkg}: MISSING -> {e}')

## 2) Create Module Skeleton in src/

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_dir = project_root / 'src' / 'asr_module'
src_dir.mkdir(parents=True, exist_ok=True)

(src_dir / '__init__.py').write_text(
    'from .asr import ASRModule\n\n__all__ = ["ASRModule"]\n',
    encoding='utf-8'
)

if not (src_dir / 'asr.py').exists():
    (src_dir / 'asr.py').write_text('# ASR module placeholder\n', encoding='utf-8')

print('Created:', src_dir)
print('Files:', [p.name for p in src_dir.iterdir()])

## 3) Implement ASR Core Class for Vietnamese

Cell dưới đây ghi class `ASRModule` vào `src/asr_module/asr.py` với các method:
- `test_inference()`
- `realtime_inference()`
- utility methods cho resample/chunk/postprocess.

In [ ]:
asr_py = '''
from __future__ import annotations

import queue
import re
import threading
import time
from pathlib import Path
from typing import Callable, Optional

import librosa
import numpy as np

from modules.asr_zipformer import VietnameseZipformerASR


class ASRModule:
    """Vietnamese ASR wrapper with test and realtime inference."""

    def __init__(
        self,
        model_dir: str | Path,
        device: str = 'cpu',
        sample_rate: int = 16000,
        chunk_seconds: float = 0.6,
        overlap_seconds: float = 0.1,
    ) -> None:
        self.model_dir = Path(model_dir)
        self.device = device
        self.sample_rate = sample_rate
        self.chunk_seconds = chunk_seconds
        self.overlap_seconds = overlap_seconds

        self.backend = VietnameseZipformerASR(
            model_dir=self.model_dir,
            sample_rate=self.sample_rate,
            provider=device,
        )

    def _resample_if_needed(self, samples: np.ndarray, sr: int) -> np.ndarray:
        if sr == self.sample_rate:
            return samples.astype(np.float32)
        return librosa.resample(
            samples.astype(np.float32),
            orig_sr=sr,
            target_sr=self.sample_rate,
        ).astype(np.float32)

    def _chunk_audio(self, samples: np.ndarray) -> list[np.ndarray]:
        chunk_size = int(self.chunk_seconds * self.sample_rate)
        overlap = int(self.overlap_seconds * self.sample_rate)
        if chunk_size <= 0:
            raise ValueError('chunk_seconds must be > 0')

        chunks: list[np.ndarray] = []
        step = max(1, chunk_size - overlap)
        for start in range(0, len(samples), step):
            end = start + chunk_size
            chunk = samples[start:end]
            if len(chunk) == 0:
                continue
            chunks.append(chunk)
            if end >= len(samples):
                break
        return chunks

    def _postprocess_text(self, text: str) -> str:
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'\s+([,.;:!?])', r'\1', text)
        return text

    def test_inference(self, audio_path: str | Path) -> str:
        audio_path = Path(audio_path)
        samples, sr = librosa.load(audio_path, sr=None, mono=True)
        samples = self._resample_if_needed(samples, sr)
        text = self.backend.transcribe_array(samples, self.sample_rate)
        return self._postprocess_text(text)

    def realtime_inference(
        self,
        duration_seconds: Optional[float] = 10.0,
        on_partial_result: Optional[Callable[[str], None]] = None,
        on_final_result: Optional[Callable[[str], None]] = None,
        print_partial: bool = True,
    ) -> str:
        return self.backend.realtime_inference(
            duration_seconds=duration_seconds,
            on_partial_result=on_partial_result,
            on_final_result=on_final_result,
            print_partial=print_partial,
        )
'''

(src_dir / 'asr.py').write_text(asr_py, encoding='utf-8')
print('Wrote:', src_dir / 'asr.py')

## 4) Implement test_inference() for Audio File Transcription

Chạy cell sau để import class mới từ `src/` và chuẩn bị cấu hình model.

In [ ]:
import sys

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from asr_module import ASRModule

MODEL_DIR = project_root / 'models' / 'zipformer-vi'  # đổi theo thư mục model thực tế
SAMPLE_AUDIO = project_root / 'data' / 'raw' / 'sample_vi.wav'  # đổi theo file thực tế

print('MODEL_DIR:', MODEL_DIR)
print('SAMPLE_AUDIO:', SAMPLE_AUDIO)

## 5) Implement realtime_inference() for Microphone Streaming

Method `realtime_inference()` đã được định nghĩa trong class ở phần trên. Cell dưới đây khởi tạo module.

In [ ]:
asr = ASRModule(
    model_dir=MODEL_DIR,
    device='cpu',
    sample_rate=16000,
    chunk_seconds=0.6,
    overlap_seconds=0.1,
)
print('ASRModule initialized')

## 6) Add Utility Functions (Resampling, Chunking, Text Post-processing)

Các utility đã nằm trong class:
- `_resample_if_needed`
- `_chunk_audio`
- `_postprocess_text`

Cell dưới đây test nhanh utility chunking.

In [ ]:
fake_audio = np.random.randn(16000 * 3).astype(np.float32)
chunks = asr._chunk_audio(fake_audio)
print('Num chunks:', len(chunks))
print('First chunk shape:', chunks[0].shape if chunks else None)

## 7) Run File-Based Smoke Tests

In [ ]:
if SAMPLE_AUDIO.exists():
    result = asr.test_inference(SAMPLE_AUDIO)
    print('Transcript:', result)
else:
    print('Sample audio not found. Update SAMPLE_AUDIO path before running smoke test.')

## 8) Run Realtime Inference Demo in Notebook

Lưu ý:
- Cấp quyền microphone cho kernel/Jupyter.
- Nên dùng sample rate 16000 Hz.
- Có thể bấm stop trong notebook hoặc Ctrl+C nếu chạy terminal.

In [ ]:
# Realtime demo trong 15 giây
# transcript_rt = asr.realtime_inference(duration_seconds=15)
# print('\nRealtime transcript:', transcript_rt)

print('Uncomment the lines above to run realtime ASR.')

In [ ]:
partials = []
finals = []

def on_partial(text: str):
    partials.append(text)

def on_final(text: str):
    finals.append(text)

# Callback-based demo (optional)
# asr.realtime_inference(
#     duration_seconds=10,
#     on_partial_result=on_partial,
#     on_final_result=on_final,
#     print_partial=False,
# )
# print('Final chunks:', finals)

print('Callback demo is ready. Uncomment to run.')

## 9) Update requirements.txt Programmatically

In [ ]:
requirements_path = project_root / 'requirements.txt'
missing = [
    'sherpa-onnx>=1.10.40',
    'sounddevice>=0.4.7',
    'librosa>=0.10.2',
    'numpy>=1.24.0',
]

existing_lines = requirements_path.read_text(encoding='utf-8').splitlines()
existing_pkgs = {line.split('>=')[0].strip() for line in existing_lines if line and not line.strip().startswith('#')}

to_add = [line for line in missing if line.split('>=')[0] not in existing_pkgs]
if to_add:
    with requirements_path.open('a', encoding='utf-8') as f:
        f.write('\n# ASR notebook extras\n')
        for dep in to_add:
            f.write(dep + '\n')

print('Added dependencies:', to_add)
print('--- requirements.txt ---')
print(requirements_path.read_text(encoding='utf-8'))

## 10) Validate End-to-End Module Usage

In [ ]:
from modules.asr_zipformer import VietnameseZipformerASR

print('Import from modules.asr_zipformer: OK')

if MODEL_DIR.exists():
    backend = VietnameseZipformerASR(model_dir=MODEL_DIR, sample_rate=16000)
    print('Backend init: OK')
    if SAMPLE_AUDIO.exists():
        print('File test:', backend.transcribe_wave_file(SAMPLE_AUDIO))
    else:
        print('Skip file test: SAMPLE_AUDIO does not exist.')

    # Uncomment for a short realtime run:
    # print(backend.realtime_inference(duration_seconds=5))
else:
    print('Skip validation: MODEL_DIR does not exist. Update MODEL_DIR first.')